# RAY-IMAGE v0.1 — First Working Prototype

This notebook trains the prototype in two stages and evaluates whether text conditioning affects the generated image.

Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select a GPU runtime and reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!pip install -q -r requirements.txt

In [ ]:
!python -m ray_image.train_smoke

In [ ]:
!python tools/make_toy_dataset.py --output data/toy --samples 256 --size 64

In [ ]:
!python -m ray_image.train_vae --manifest data/toy/manifest.jsonl --steps 1000 --batch-size 16 --save /content/ray_vae_v0_1.pt

In [ ]:
!python -m ray_image.train_generator --manifest data/toy/manifest.jsonl --vae /content/ray_vae_v0_1.pt --steps 2000 --batch-size 8 --save /content/ray_image_v0_1_trained.pt

In [ ]:
!python -m ray_image.generate --checkpoint /content/ray_image_v0_1_trained.pt --prompt 'a blue circle' --steps 40 --seed 42 --output /content/first_rays_image.png

In [ ]:
from pathlib import Path
from PIL import Image
image_path = Path('/content/first_rays_image.png')
print('generated:', image_path.exists())
print('file_size_bytes:', image_path.stat().st_size if image_path.exists() else '-')
if image_path.exists():
    im = Image.open(image_path)
    print('dimensions:', im.size)

In [ ]:
!python -m ray_image.evaluate /content/first_rays_image.png

In [ ]:
# Generate a small prompt suite so we can compare text-conditioned outputs numerically.
from pathlib import Path
prompts = [
    ('blue_circle', 'a blue circle'),
    ('red_circle', 'a red circle'),
    ('blue_square', 'a blue square'),
    ('yellow_triangle', 'a yellow triangle'),
]
for name, prompt in prompts:
    out = Path('/content') / f'{name}.png'
    !python -m ray_image.generate --checkpoint /content/ray_image_v0_1_trained.pt --prompt "{prompt}" --steps 40 --seed 42 --output "{out}"
    print(f'created {name}:', out.exists(), 'bytes=', out.stat().st_size if out.exists() else '-')

In [ ]:
!python -m ray_image.evaluate /content/blue_circle.png /content/red_circle.png /content/blue_square.png /content/yellow_triangle.png

## Interpretation

The diagnostics are only a toy-experiment signal, not a perceptual quality score. We want the requested color/shape changes to produce measurably different outputs. If they do not, the next training change is to strengthen text conditioning and train longer.